In [0]:
%pip install sqlalchemy psycopg2-binary tqdm

In [0]:
import pandas as pd
import numpy as np
import psycopg2
from sqlalchemy import create_engine
from tqdm import tqdm

RDS_HOST = dbutils.secrets.get(scope="aws-credentials", key="rds-host")
RDS_USER = dbutils.secrets.get(scope="aws-credentials", key="rds-user")
RDS_PASS = dbutils.secrets.get(scope="aws-credentials", key="rds-password")
RDS_DB   = "enem_db"
RDS_PORT = 5432

engine = create_engine(
    f"postgresql+psycopg2://{RDS_USER}:{RDS_PASS}@{RDS_HOST}:{RDS_PORT}/{RDS_DB}"
)
print("aqui foi 2")

In [0]:
#mapeamento
MAPA_SEXO = {"M": "Masculino", "F": "Feminino"}
MAPA_COR  = {"0": "Não declarado", "1": "Branca", "2": "Preta",
              "3": "Parda", "4": "Amarela", "5": "Indígena"}
MAPA_ESCOLA = {"1": "Não respondeu", "2": "Pública",
                "3": "Privada",       "4": "Exterior"}
MAPA_FAIXA  = {
    "1": "Menor de 17", "2": "17 anos", "3": "18 anos",
    "4": "19 anos", "5": "20 anos", "6": "21 anos",
    "7": "22 anos", "8": "23 anos", "9": "24 anos",
    "10": "25 anos", "11": "Entre 26 e 30",
    "12": "Entre 31 e 35", "13": "Entre 36 e 40",
    "14": "Entre 41 e 45", "15": "Entre 46 e 50",
    "16": "Entre 51 e 55", "17": "56 anos ou mais"
}

NOTAS = ["nu_nota_cn", "nu_nota_ch", "nu_nota_lc", "nu_nota_mt", "nu_nota_redacao"]

def transform_chunk(chunk):
    # notas to float 
    for col in NOTAS:
        chunk[col] = chunk[col].astype(str).str.replace(",", ".").str.strip()
        chunk[col] = pd.to_numeric(chunk[col], errors="coerce")
        chunk[col] = chunk[col].replace(0.0, np.nan)

    # calcular medias
    chunk["nu_media_objetivas"] = chunk[["nu_nota_cn","nu_nota_ch",
                                          "nu_nota_lc","nu_nota_mt"]].mean(axis=1)
    chunk["nu_media_total"]     = chunk[NOTAS].mean(axis=1)

    # labels
    chunk["ds_sexo"]        = chunk["tp_sexo"].map(MAPA_SEXO)
    chunk["ds_cor_raca"]    = chunk["tp_cor_raca"].astype(str).map(MAPA_COR)
    chunk["ds_escola"]      = chunk["tp_escola"].astype(str).map(MAPA_ESCOLA)
    chunk["ds_faixa_etaria"]= chunk["tp_faixa_etaria"].astype(str).map(MAPA_FAIXA)

    # tipagem correta
    chunk["nu_ano"]                  = pd.to_numeric(chunk["nu_ano"], errors="coerce").astype("Int64")
    chunk["tp_cor_raca"]             = pd.to_numeric(chunk["tp_cor_raca"], errors="coerce").astype("Int64")
    chunk["tp_escola"]               = pd.to_numeric(chunk["tp_escola"], errors="coerce").astype("Int64")
    chunk["tp_faixa_etaria"]         = pd.to_numeric(chunk["tp_faixa_etaria"], errors="coerce").astype("Int64")
    chunk["co_uf_residencia"]        = pd.to_numeric(chunk["co_uf_residencia"], errors="coerce").astype("Int64")
    chunk["co_municipio_residencia"] = pd.to_numeric(chunk["co_municipio_residencia"], errors="coerce").astype("Int64")
    chunk["in_treineiro"]            = chunk["in_treineiro"].map({"0": False, "1": True})

    # selecao de colunas do schema tru
    colunas_tru = [
        "nu_inscricao", "nu_ano", "tp_sexo", "ds_sexo",
        "tp_cor_raca", "ds_cor_raca", "tp_escola", "ds_escola",
        "tp_faixa_etaria", "ds_faixa_etaria", "in_treineiro",
        "co_uf_residencia", "sg_uf_residencia",
        "co_municipio_residencia", "no_municipio_residencia",
        "nu_nota_cn", "nu_nota_ch", "nu_nota_lc", "nu_nota_mt",
        "nu_nota_redacao", "nu_media_objetivas", "nu_media_total",
        "tp_status_redacao", "q001", "q002", "q006"
    ]
    colunas_existentes = [c for c in colunas_tru if c in chunk.columns]
    return chunk[colunas_existentes]

# processar em chuncks direto para a tru
print("Iniciando ETL RAW pra TRU")
CHUNK_SIZE = 50_000 #cargas de 50k em 50k para evitar timeouts
offset     = 0
total      = 0
first      = True

while True:
    query = f"""
        SELECT * FROM raw.enem_microdados
        LIMIT {CHUNK_SIZE} OFFSET {offset}
    """
    chunk = pd.read_sql(query, engine)

    if chunk.empty:
        break

    chunk_tru = transform_chunk(chunk)
    chunk_tru = chunk_tru.drop_duplicates(subset=["nu_inscricao"])

    chunk_tru.to_sql(
        "enem_candidatos",
        engine,
        schema="tru",
        if_exists="replace" if first else "append",
        index=False,
        method="multi",
        chunksize=1000
    )

    first   = False
    offset += CHUNK_SIZE
    total  += len(chunk_tru)
    print(f"{total:,} registros na TRU")

    if len(chunk) < CHUNK_SIZE:
        break

print(f"\nETL feito — {total:,} registros gravados na tabela tru.enem_candidatos")